# 02 — Data Ingestion (Bab 2.2)Notebook ini mengimplementasikan proses **Batch Ingestion** — memuat dataset CSV ke dalam Spark DataFrame menggunakan **explicit schema** (sesuai best practice).

## 2.1 Inisialisasi SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("02_DataIngestion") \
    .master("local[*]") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

print(f"✅ Spark {spark.version} connected")

✅ Spark 3.5.0 connected


## 2.2 Definisi Explicit SchemaSesuai RULES: *"Always define explicit schema, avoid schema inference in large datasets"*

In [2]:
from pyspark.sql.types import (
    StructType, StructField, 
    IntegerType, StringType
)

# Schema eksplisit untuk retail_sales_dataset.csv
schema = StructType([
    StructField("Transaction_ID", IntegerType(), nullable=False),
    StructField("Date", StringType(), nullable=False),
    StructField("Customer_ID", StringType(), nullable=False),
    StructField("Gender", StringType(), nullable=False),
    StructField("Age", IntegerType(), nullable=False),
    StructField("Product_Category", StringType(), nullable=False),
    StructField("Quantity", IntegerType(), nullable=False),
    StructField("Price_per_Unit", IntegerType(), nullable=False),
    StructField("Total_Amount", IntegerType(), nullable=False)
])

print("✅ Schema didefinisikan:")
for field in schema.fields:
    print(f"   {field.name:20s} → {str(field.dataType):15s} (nullable={field.nullable})")

✅ Schema didefinisikan:
   Transaction_ID       → IntegerType()   (nullable=False)
   Date                 → StringType()    (nullable=False)
   Customer_ID          → StringType()    (nullable=False)
   Gender               → StringType()    (nullable=False)
   Age                  → IntegerType()   (nullable=False)
   Product_Category     → StringType()    (nullable=False)
   Quantity             → IntegerType()   (nullable=False)
   Price_per_Unit       → IntegerType()   (nullable=False)
   Total_Amount         → IntegerType()   (nullable=False)


## 2.3 Batch Ingestion — Load CSV

In [3]:
# Load CSV dengan explicit schema
df_raw = spark.read.csv(
    "/data/retail_sales_dataset.csv",
    header=True,
    schema=schema
)

# Verifikasi hasil ingestion
print(f"✅ Data berhasil di-load!")
print(f"   Jumlah baris   : {df_raw.count()}")
print(f"   Jumlah kolom   : {len(df_raw.columns)}")
print(f"   Partisi        : {df_raw.rdd.getNumPartitions()}")
print()
print("=== Schema ===")
df_raw.printSchema()

✅ Data berhasil di-load!
   Jumlah baris   : 1000
   Jumlah kolom   : 9
   Partisi        : 1

=== Schema ===
root
 |-- Transaction_ID: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Price_per_Unit: integer (nullable = true)
 |-- Total_Amount: integer (nullable = true)



## 2.4 Preview Data

In [4]:
# Tampilkan 10 baris pertama
df_raw.show(10, truncate=False)

+--------------+----------+-----------+------+---+----------------+--------+--------------+------------+
|Transaction_ID|Date      |Customer_ID|Gender|Age|Product_Category|Quantity|Price_per_Unit|Total_Amount|
+--------------+----------+-----------+------+---+----------------+--------+--------------+------------+
|1             |2023-11-24|CUST001    |Male  |34 |Beauty          |3       |50            |150         |
|2             |2023-02-27|CUST002    |Female|26 |Clothing        |2       |500           |1000        |
|3             |2023-01-13|CUST003    |Male  |50 |Electronics     |1       |30            |30          |
|4             |2023-05-21|CUST004    |Male  |37 |Clothing        |1       |500           |500         |
|5             |2023-05-06|CUST005    |Male  |30 |Beauty          |2       |50            |100         |
|6             |2023-04-25|CUST006    |Female|45 |Beauty          |1       |30            |30          |
|7             |2023-03-13|CUST007    |Male  |46 |Cloth

## 2.5 Statistik Deskriptif Awal

In [5]:
# Summary statistics untuk kolom numerik
df_raw.select("Age", "Quantity", "Price_per_Unit", "Total_Amount").summary().show()

+-------+------------------+------------------+------------------+-----------------+
|summary|               Age|          Quantity|    Price_per_Unit|     Total_Amount|
+-------+------------------+------------------+------------------+-----------------+
|  count|              1000|              1000|              1000|             1000|
|   mean|            41.392|             2.514|            179.89|            456.0|
| stddev|13.681429659122518|1.1327343409145354|189.68135627129234|559.9976315551235|
|    min|                18|                 1|                25|               25|
|    25%|                29|                 1|                30|               60|
|    50%|                42|                 3|                50|              120|
|    75%|                53|                 4|               300|              900|
|    max|                64|                 4|               500|             2000|
+-------+------------------+------------------+------------------

## 2.6 Simpan DataFrame sebagai Temporary ViewDisimpan agar bisa diakses di notebook selanjutnya via Spark SQL.

In [6]:
df_raw.createOrReplaceTempView("raw_transactions")
print("✅ Temporary view 'raw_transactions' dibuat")
print("   Bisa diakses dengan: spark.sql('SELECT * FROM raw_transactions')")

✅ Temporary view 'raw_transactions' dibuat
   Bisa diakses dengan: spark.sql('SELECT * FROM raw_transactions')


In [7]:
# Jangan stop spark — akan dipakai di notebook berikutnya
print("📌 SparkSession tetap aktif untuk notebook selanjutnya")

📌 SparkSession tetap aktif untuk notebook selanjutnya
